In [0]:
from pyspark.sql import functions as F
import logging
import sys
from pyspark.sql.types import StructType, StructField, TimestampType, IntegerType, FloatType
import uuid
from datetime import datetime, timezone

logger = logging.getLogger("turbines")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)
logger.propagate = False

run_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc)

logger.info(f"Bronze started at {started_at} with pipeline id {run_id}")

In [0]:
def log_run(layer, started_at, rows_in, rows_out, status="SUCCESS", message=None):
    spark.createDataFrame(
        [(run_id, layer, started_at, datetime.now(timezone.utc), rows_in, rows_out, status, message)],
        "run_id string, layer string, started_at timestamp, finished_at timestamp, rows_in long, rows_out long, status string, message string"
    ).write.mode("append").saveAsTable("turbines.control.pipeline_log")

In [0]:
schema = StructType([
    StructField("timestamp",      TimestampType(), True),
    StructField("turbine_id",     IntegerType(),   True),
    StructField("wind_speed",     FloatType(),     True),
    StructField("wind_direction", FloatType(),     True),
    StructField("power_output",   FloatType(),     True),
])

In [0]:
storage_account = "sacuksnprdcdproject0001"

corrupted_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/project0001/nprd/turbine/landing_corrupted/"

df = spark.read.option("header", True).schema(schema).csv(corrupted_path)
n_in = df.count()
logger.info(f"Read {n_in} records from {corrupted_path}")

In [0]:
df_bronze = (df
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("ingest_ts", F.current_timestamp())
)

In [0]:
df_bronze.write.mode("overwrite").saveAsTable("turbines.bronze.raw")

In [0]:

n_out = spark.table("turbines.bronze.raw").count()  
log_run("bronze", started_at, n_in, n_out)
spark.table("turbines.control.pipeline_log").show()